## **Step 0: Import Libraries**

In [1]:
# Data Handling
import pandas as pd


# Supressing Unnecessary Warnings
import warnings

# File Handling
import os
from pathlib import Path

# Supressing Warnings
warnings.simplefilter("ignore")
# Show all columns
pd.set_option("display.max_columns", None)


In [2]:
# Get current working directory
cwd = os.getcwd()
print("Current working directory:", cwd)

# Change working directory to project root
os.chdir("../")
print("Changed working directory to root:", os.getcwd())


Current working directory: d:\MLops\PowerCo-Custumer-Churn\Notebooks
Changed working directory to root: d:\MLops\PowerCo-Custumer-Churn


In [3]:
CLIENTS_CSV = Path("data/client_data.csv")
PRICES_CSV = Path("data/price_data.csv")
print("Clients CSV Path:", CLIENTS_CSV)
print("Prices CSV Path:", PRICES_CSV)

Clients CSV Path: data\client_data.csv
Prices CSV Path: data\price_data.csv


## **Step 1: Load Data**

In [4]:
date_cols_clients = ["date_activ", "date_end", "date_modif_prod", "date_renewal"]
clients = pd.read_csv(CLIENTS_CSV, parse_dates=date_cols_clients, low_memory=False)
prices = pd.read_csv(PRICES_CSV, parse_dates=["price_date"], low_memory=False)


display(clients.head())
print("================" * 50)
display(prices.head())

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_pow_off_peak,has_gas,imp_cons,margin_gross_pow_ele,margin_net_pow_ele,nb_prod_act,net_margin,num_years_antig,origin_up,pow_max,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,0,0.0,1.78,0.114481,0.098142,40.606701,t,0.00,25.44,25.44,2,678.99,3,lxidpiddsbxsbosboudacockeimpuepw,43.648,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,0,0.0,16.27,0.145711,0.000000,44.311378,f,0.00,16.38,16.38,1,18.89,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.800,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,0,0.0,38.72,0.165794,0.087899,44.311378,f,0.00,28.60,28.60,1,6.60,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.856,0
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,0,0.0,19.83,0.146694,0.000000,44.311378,f,0.00,30.22,30.22,1,25.46,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.200,0
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,526,0.0,131.73,0.116900,0.100015,40.606701,f,52.32,44.91,44.91,1,47.98,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,19.800,0


,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


### **Check for missing and duplicated rows**

In [5]:
# Check for missing values in prices
missing_price = prices.isnull().sum()
print("Missing values in prices:")
print(missing_price[missing_price > 0])
print("================")
missing_client = clients.isnull().sum()
print("Missing values in clients:")
print(missing_client[missing_client > 0])
print("================")
# Check for duplicates in prices
duplicates_price = prices.duplicated().sum()
print(f"Number of duplicate rows in prices: {duplicates_price}")
print("================")
# Check for duplicates in clients
duplicates_client = clients.duplicated().sum()
print(f"Number of duplicate rows in clients: {duplicates_client}")
print("================")


Missing values in prices:
Series([], dtype: int64)
Missing values in clients:
Series([], dtype: int64)
Number of duplicate rows in prices: 0
Number of duplicate rows in clients: 0


## **Step 2: Aggregate Price Data (Customer-Level Summary)**

### **Ensuring consistent monthly bucketing**
- This line creates a new column called "month" in the prices DataFrame by transforming each price_date into the `last day of its month`.This ensures consistent monthly bucketing, which is essential for:
   - Aggregating price behavior by month
   - Aligning features with churn prediction windows
   - Enforcing leakage-safe cutoffs (e.g., using data ≤ m–1)

In [6]:
# Normalize price_date to month-end
prices["month"] = prices["price_date"] + pd.offsets.MonthEnd(0)
print(prices[["price_date", "month"]].head(3))


  price_date      month
0 2015-01-01 2015-01-31
1 2015-02-01 2015-02-28
2 2015-03-01 2015-03-31


### **Build a per-customer reference month (one-snapshot path)**
- This line computes the latest month of price activity for each customer and merges it to clients Keeping all rows from clients, and adds last_price_month where available.

In [ ]:
price_cols = [
    "price_off_peak_var",
    "price_peak_var",
    "price_mid_peak_var",
    "price_off_peak_fix",
    "price_peak_fix",
    "price_mid_peak_fix",
]

# Collapse to one row per (id, month) by averaging if multiples exist
prices_m = (
    prices.groupby(["id", "month"], as_index=False)[price_cols]
    .mean()
    .sort_values(["id", "month"])
)

# For each customer, define a leakage-safe reference month m_ref (the latest month we’ll “act” from)
# This is the last month we have a price for each customer
last_price_month = (
    prices_m.groupby("id", as_index=False)["month"]
    .max()
    .rename(columns={"month": "last_price_month"})
)
# Merge last_price_month into clients to get the reference month for each client
clients_ref = clients.merge(last_price_month, on="id", how="left")


### **Create a new column called "m_ref" in the panel DataFrame.**
- It represents the cutoff month for feature generation per customer. It is the last month we can use to compute features without violating temporal integrity.
- This ensures that:
   - We don’t use price data after the renewal date, which could leak future info.
   - We don’t use data beyond what’s available in the price dataset.

In [13]:
# If date_renewal is NaT, use last_price_month; else take the earlier of the two dates
clients_ref["m_ref"] = clients_ref["date_renewal"]

# Ensure date columns are in datetime format for safety
clients_ref["m_ref"] = pd.to_datetime(clients_ref["m_ref"], errors="coerce")
clients_ref["last_price_month"] = pd.to_datetime(
    clients_ref["last_price_month"], errors="coerce"
)

# Where m_ref is NaT, replace with last_price_month
mask = clients_ref["m_ref"].isna()
clients_ref.loc[mask, "m_ref"] = clients_ref.loc[mask, "last_price_month"]
# Where both dates exist, take the earlier one
clients_ref["m_ref"] = clients_ref[["m_ref", "last_price_month"]].min(axis=1)

# Keep only customers with any usable price history + a defined m_ref
clients_ref = clients_ref.dropna(subset=["m_ref", "last_price_month"]).copy()

# Define the *feature cutoff* month = m_ref - 1 month (we will only use prices up to and including this)
clients_ref["cutoff_month"] = clients_ref["m_ref"] - pd.offsets.MonthEnd(1)

print("NaT counts after conversion:")
print(f"m_ref: {clients_ref['m_ref'].isna().sum()}")
print(f"cutoff_month: {clients_ref['cutoff_month'].isna().sum()}")


NaT counts after conversion:
m_ref: 0
cutoff_month: 0


### **Merges the m_ref (reference month) from the clients_ref back into the prices DataFrame.**
- This filters the prices DataFrame to include only the price records that occurred before the reference month (m_ref) for each customer. This prevents using future data (after renewal/decision point) for feature engineering.
- This ensures the leakage-safe feature horizon:
    - For each customer, we now have a filtered price history that ends at m_ref – 1 month.
    - We can safely compute rolling averages, volatility, deltas, etc., knowing we’re not using any data from or after the decision point.

In [14]:
# Attach the cutoff to prices and filter per-id (prevents future leakage)
prices_cut = prices_m.merge(clients_ref[["id", "cutoff_month"]], on="id", how="inner")
prices_cut = prices_cut[prices_cut["month"] <= prices_cut["cutoff_month"]].copy()

# Report coverage after enforcing time-awareness
n_clients = clients["id"].nunique()
n_prices_ids = prices["id"].nunique()
n_panel_ids = prices_cut["id"].nunique()
n_model_ids = clients_ref["id"].nunique()

print(
    {
        "unique_client_ids_in_clients": n_clients,
        "unique_ids_in_prices": n_prices_ids,
        "unique_ids_with_price_history_pre_cutoff": n_panel_ids,
        "unique_clients_with_defined_m_ref": n_model_ids,
    }
)


{'unique_client_ids_in_clients': 14606, 'unique_ids_in_prices': 16096, 'unique_ids_with_price_history_pre_cutoff': 14217, 'unique_clients_with_defined_m_ref': 14606}


- `Total Clients`: 14,606 unique IDs in clients—this is our full dataset.
- `Prices Coverage`: 16,096 unique IDs in prices, which is higher than clients. This suggests some IDs in prices aren't in 
clients (or vice versa), possibly due to data mismatches or extra entries.
- `Usable Price History`: After enforcing the cutoff (leakage-safe filtering), only 14,217 unique IDs have price data pre-cutoff. This means ~389 clients (14,606 - 14,217) lack sufficient historical price data for feature engineering—they may need imputation, exclusion, or further investigation.
- `m_ref Coverage`: All 14,606 clients have a defined m_ref, which is great for consistency.

In [10]:
# Save pre-processed data for more feature engineering
clients_ref.to_csv("Data/clients_ref.csv", index=False)
prices_cut.to_csv("Data/prices_cut.csv", index=False)